# Baseline stream recommender

This notebook consumes `data/live_stream_features.jsonl`, normalizes the basic stream features, applies an explicit linear weight vector, and produces the top 50 live streams.

There are not yet interaction labels such as clicks or watch duration, so this is a coefficient-defined linear scoring baseline rather than a statistically fitted regression. The resulting `y_score` is the target to replace with a learned target when labels are available.

## Load exported data

Run `00_get_data.ipynb` first so that `data/live_stream_features.jsonl` exists.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

notebook_dir = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
data_path = notebook_dir / 'data' / 'live_stream_features.jsonl'
if not data_path.exists():
    raise FileNotFoundError(f'Run 00_get_data.ipynb first: {data_path}')

streams_df = pd.read_json(data_path, lines=True)
print(f'Loaded {len(streams_df)} live streams from {data_path}')
streams_df.head(3)

Loaded 5000 live streams from /Users/edwynzhou/Documents/repos/TwitchEE/notebooks/data/live_stream_features.jsonl


,id,user_id,user_login,user_name,game_id,game_name,title,language,tags,is_mature,viewer_count,started_at,stream_age_hours,viewer_log1p,tag_count,title_length,is_followed_channel,matches_recent_vod_category
0,318188581220,92038375,caedrel,Caedrel,21779,League of Legends,🔴WORLDS 2026 KOI VS KC ELIMINATION🔴,en,"[English, LPL, LCK, LEC, Worlds, Esports, Drop...",False,137490,2026-10-31 13:00:36+00:00,4.129866,11.831314,7,35,False,False
1,318190806372,27115917,kamet0,Kamet0,21779,League of Legends,KARMINE CORP 0 vs 0 MOVISTAR KOI | WORLDS 2026...,fr,"[Français, LEC, leagueoflegends, esport, RLCS,...",False,109134,2026-10-31 16:20:11+00:00,0.803477,11.600341,10,108,False,False
2,318190614756,83232866,ibai,ibai,21779,League of Legends,#WORLDS2026 | MKOI vs KC | EL PARTIDO DEL AÑO ...,es,"[Español, IRL, Chatting, KOI, DropsActivados]",False,66175,2026-10-31 16:03:29+00:00,1.081811,11.100073,5,112,False,False


## Define and normalize the baseline features

All features are normalized to `[0, 1]` before weights are applied. Viewer count uses the API export's `viewer_log1p` feature so very large channels do not dominate linearly. Stream age is inverted because newer streams are preferred.

In [2]:
feature_columns = [
    'viewer_log1p',
    'tag_count',
    'title_length',
    'stream_age_hours',
    'is_followed_channel',
    'matches_recent_vod_category',
]

missing_columns = [column for column in feature_columns if column not in streams_df.columns]
if missing_columns:
    raise ValueError(f'Export is missing baseline features: {missing_columns}')

numeric_features = streams_df[feature_columns].copy()
for column in feature_columns:
    numeric_features[column] = pd.to_numeric(numeric_features[column], errors='coerce').fillna(0)

scaler = MinMaxScaler()
normalized = pd.DataFrame(
    scaler.fit_transform(numeric_features),
    columns=feature_columns,
    index=streams_df.index,
)
normalized['stream_age_hours'] = 1 - normalized['stream_age_hours']
normalized = normalized.clip(0, 1)
normalized.add_suffix('_normalized').head(3)

,viewer_log1p_normalized,tag_count_normalized,title_length_normalized,stream_age_hours_normalized,is_followed_channel_normalized,matches_recent_vod_category_normalized
0,1.000000,0.7,0.250000,0.914973,0.0,0.0
1,0.976025,1.0,0.771429,0.984342,0.0,0.0
2,0.924099,0.5,0.800000,0.978538,0.0,0.0


## Apply the linear baseline

Adjust these coefficients as the baseline evolves. Positive coefficients increase recommendation score; the coefficients sum to `1.0`, keeping `y_score` interpretable on `[0, 1]`.

In [3]:
baseline_weights = {
    'viewer_log1p': 0.50,
    'tag_count': 0.05,
    'title_length': 0.05,
    'stream_age_hours': 0.05,
    'is_followed_channel': 0.20,
    'matches_recent_vod_category': 0.15,
}

if not np.isclose(sum(baseline_weights.values()), 1.0):
    raise ValueError('Baseline weights must sum to 1.0')

weights = pd.Series(baseline_weights).reindex(feature_columns)
streams_scored_df = streams_df.copy()
streams_scored_df['y_score'] = normalized.dot(weights)
streams_scored_df = streams_scored_df.sort_values(
    ['y_score', 'viewer_count'], ascending=[False, False]
)

top_50_streams_df = streams_scored_df.head(50).reset_index(drop=True)
top_50_streams_df.insert(0, 'rank', top_50_streams_df.index + 1)
top_50_streams_df[['rank', 'user_name', 'game_name', 'title', 'viewer_count', 'y_score']].head(50)

,rank,user_name,game_name,title,viewer_count,y_score
0,1,Kamet0,League of Legends,KARMINE CORP 0 vs 0 MOVISTAR KOI | WORLDS 2026...,109134,0.625801
1,2,Caedrel,League of Legends,🔴WORLDS 2026 KOI VS KC ELIMINATION🔴,137490,0.593249
2,3,ibai,League of Legends,#WORLDS2026 | MKOI vs KC | EL PARTIDO DEL AÑO ...,66175,0.575976
3,4,Trymacs,Minecraft,100.000 TNT Exploion Craft Attack Rekord aufst...,31062,0.521966
4,5,otplol_,League of Legends,WORLDS - KC vs MKOI - POUR UNE PLACE EN QUART ...,36421,0.515427
5,6,EDISON,Just Chatting,🎃КОНКУРС КОСТЮМОВ НА ХЕЛЛОУИН! [сейчас],27898,0.499949
6,7,afoninje,Dota 2,World Squad [0:0] CIS Squad | Bo1 | PARI UNIVE...,19762,0.497839
7,8,Riot Games,League of Legends,KC vs MKOI | WORLDS 2026 | Swiss Stage Day 8,28350,0.497634
8,9,Warframe,Warframe,"PMCF Quest Finale | Saturday, October 31 @ 12P...",26387,0.495889
9,10,WarThunder_Esports,War Thunder,Invictus 1-2 VOYAGER | 4v4 Ground | WTCS Pro L...,19416,0.492521


In [4]:
output_path = notebook_dir / 'data' / 'baseline_top_50.jsonl'
top_50_streams_df.to_json(output_path, orient='records', lines=True, date_format='iso')
print(f'Wrote {len(top_50_streams_df)} ranked streams to {output_path}')
print('Weights:', baseline_weights)

Wrote 50 ranked streams to /Users/edwynzhou/Documents/repos/TwitchEE/notebooks/data/baseline_top_50.jsonl
Weights: {'viewer_log1p': 0.5, 'tag_count': 0.05, 'title_length': 0.05, 'stream_age_hours': 0.05, 'is_followed_channel': 0.2, 'matches_recent_vod_category': 0.15}
